In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import matplotlib.pyplot as plt

# ==============================
# 1️⃣ CONFIGURACIÓN Y CARGA BASE
# ==============================
BASE_PATH = r"C:\Users\isaac\OneDrive\Documents\PROYECTO LUNARES\archive"
CSV_PATH = os.path.join(BASE_PATH, "HAM10000_metadata.csv")
IMG_DIR_1 = os.path.join(BASE_PATH, "HAM10000_images_part_1")
IMG_DIR_2 = os.path.join(BASE_PATH, "HAM10000_images_part_2")

df = pd.read_csv(CSV_PATH)
benign = ['nv', 'bkl', 'df', 'vasc']
df['label'] = df['dx'].apply(lambda x: 0 if x in benign else 1)

# ============================
# 2️⃣ FUNCIÓN DE CARGA IMÁGENES
# ============================
def load_image(image_id, size=(128, 128)):
    for folder in [IMG_DIR_1, IMG_DIR_2]:
        path = os.path.join(folder, f"{image_id}.jpg")
        if os.path.exists(path):
            img = Image.open(path).convert('RGB')
            img = img.resize(size)
            return np.array(img) / 255.0
    return None

# ==============================
# 3️⃣ FUNCIONES ABCDE FEATURES
# ==============================
def extract_asymmetry(image):
    gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    flipped = cv2.flip(gray, 1)
    diff = cv2.absdiff(gray, flipped)
    return np.mean(diff) / 255.0

def extract_border_irregularity(image):
    gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    return np.sum(edges) / (gray.shape[0] * gray.shape[1])

def extract_color_variety(image):
    hsv = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2HSV)
    return np.std(hsv[:, :, 0]) / 180.0  # normalizado

def extract_diameter(image):
    gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        area = cv2.contourArea(max(contours, key=cv2.contourArea))
        return np.sqrt(area) / gray.shape[0]
    return 0

def extract_elevation(image):
    # Simulación: mide contraste local como proxy de elevación
    gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    return laplacian_var / 1000.0

def extract_ABCDE_features(image):
    A = extract_asymmetry(image)
    B = extract_border_irregularity(image)
    C = extract_color_variety(image)
    D = extract_diameter(image)
    E = extract_elevation(image)
    return np.array([A, B, C, D, E])

# ==============================
# 4️⃣ CARGAR IMÁGENES Y FEATURES
# ==============================
X_imgs, X_abcd, y_labels = [], [], []

for i, row in enumerate(df.itertuples(), 1):
    img = load_image(row.image_id)
    if img is not None:
        features = extract_ABCDE_features(img)
        X_imgs.append(img)
        X_abcd.append(features)
        y_labels.append(row.label)
    if i % 500 == 0:
        print(f"→ Procesadas {i}/{len(df)} imágenes...")

X_imgs = np.array(X_imgs, dtype='float32')
X_abcd = np.array(X_abcd, dtype='float32')
y = np.array(y_labels, dtype=int)

print(f"\n✅ Imágenes: {X_imgs.shape} | Features ABCDE: {X_abcd.shape} | Etiquetas: {y.shape}")

# Escalar las features ABCDE
scaler = StandardScaler()
X_abcd = scaler.fit_transform(X_abcd)

# ============================
# 5️⃣ DIVISIÓN TRAIN/TEST
# ============================
X_img_train, X_img_test, X_feat_train, X_feat_test, y_train, y_test = train_test_split(
    X_imgs, X_abcd, y, test_size=0.2, random_state=42, stratify=y
)

# ============================
# 6️⃣ DATA AUGMENTATION
# ============================
datagen = ImageDataGenerator(
    rotation_range=15,
    horizontal_flip=True,
    vertical_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)
datagen.fit(X_img_train)

# ============================
# 7️⃣ MODELO CNN + ABCDE
# ============================
input_img = layers.Input(shape=(128, 128, 3))
input_feat = layers.Input(shape=(5,))

x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(input_img)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(64, activation='relu')(x)

# Concatenar las features ABCDE
combined = layers.concatenate([x, input_feat])
z = layers.Dense(64, activation='relu')(combined)
z = layers.Dropout(0.3)(z)
output = layers.Dense(1, activation='sigmoid')(z)

model = models.Model(inputs=[input_img, input_feat], outputs=output)

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='AUC')]
)

model.summary()

# ============================
# 8️⃣ CALLBACKS
# ============================
callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.3, patience=5),
    ModelCheckpoint('mejor_modelo_lunares.h5', save_best_only=True)
]

# ============================
# 9️⃣ ENTRENAMIENTO
# ============================
history = model.fit(
    datagen.flow([X_img_train, X_feat_train], y_train, batch_size=32),
    validation_data=([X_img_test, X_feat_test], y_test),
    epochs=50,
    callbacks=callbacks
)

# ============================
# 🔟 EVALUACIÓN Y MÉTRICAS
# ============================
y_pred = (model.predict([X_img_test, X_feat_test]) > 0.5).astype(int)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
plt.title("Matriz de confusión")
plt.imshow(cm, cmap='Blues')
plt.xlabel("Predicho")
plt.ylabel("Real")
plt.colorbar()
plt.show()

# Curvas
plt.figure(figsize=(10,5))
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.title('Evolución de accuracy')
plt.legend()
plt.show()

plt.figure(figsize=(10,5))
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.title('Evolución de pérdida')
plt.legend()
plt.show()
